In [1]:

from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Trained adapter path
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v5" 

print("Loading model...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 1},
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("Model loaded successfully!")

Loading model...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.21it/s]


Tokenizer vocab size: 128256


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Model loaded successfully!


In [3]:
# Test the model with the CORRECT tokenizer (the one with special tokens)
def test_model(patient_id, query, max_length=1024):
    # Format exactly like training data - include the explicit instruction
    messages = [
        {
            "role": "system",
            "content": "You are a compassionate mental health support assistant who responds in a warm, empathetic, and non-judgmental way, validating the user’s feelings, offering practical and evidence-based coping suggestions, encouraging professional help when appropriate, providing crisis resources only if there is self-harm or suicide risk, and tailoring hopeful guidance to the user’s specific situation."
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.1,              
            # top_p=0.95,                 
            # repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            
            
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

In [5]:
# Test with the loaded model and tokenizer
test_query = "Our relationship ended about 7 years ago, but don't know how to let go. How can I get over that person and move on?"
response = test_model("UID_123", test_query)
# print(f"Response: \n{response}")
print(response)
print("="*80)

It's never easy to move on from a relationship, especially when it was a significant part of your life for a number of years. It's also possible that you may have been single for a number of years and then entered into this relationship. In either case, there are a number of things that may be making it difficult for you to "let go." First and foremost, it's important to acknowledge that it's normal to feel sad and to grieve the loss of the relationship. You may also be clinging to the past and holding onto "what could have beens." It's important to accept the loss of the relationship and to understand that the relationship did not work out for the best. It's also important to look at the lessons that you can take with you and apply to your current relationship. It may also be helpful to find a support group or a therapist to help you to work through these feelings and to gain some new skills that will help you to have a satisfying relationship in the future.As for practical tips, try 